In [1]:
!pip install geopy pandas tqdm

   ---------------------------------------- 0.0/125.4 kB ? eta -:--:--
   ---------------------------------------- 125.4/125.4 kB 2.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/40.7 kB ? eta -:--:--
   ---------------------------------------- 40.7/40.7 kB 1.9 MB/s eta 0:00:00


In [13]:
import geopandas as gpd
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from tqdm import tqdm

In [9]:
gdf = gpd.read_file(r'C:\Users\gabriel.coimbra\Downloads\Help Manu\E_EEE.gpkg')

In [11]:
gdf

,SUB_BACIA,DISTRITO,ETAPA_CICLO,REGIONAL,UF,UN,MUNICIPIO,BAIRRO,NOME,NOTAS,...,LAST_EDITED_DATE,SUPERINTENDENCIA,GRUPO_EXECUTIVO,BACIA,ELEVACAO_TERRENO,CAPACIDADE_M3,NIVEL_MINIMO,NIVEL_MAXIMO,ELEVACAO_FUNDO,geometry
0,MB-AS5,Sede,Projeto Conceitual,None,None,None,None,None,EEB AS5,None,...,NaT,None,None,MB-AS5,178.000,"0,144799683",175.718060,175.800,175.118060,POINT Z (363779.857 6717729.388 0)
1,MB-AS4,Sede,Projeto Conceitual,None,None,None,None,None,EEB AS4-1,None,...,NaT,None,None,MB-AS4,191.000,"0,905387233",188.215556,188.400,187.465556,POINT Z (364993.158 6717356.479 0)
2,MB-AS1-B,Sede,Projeto Conceitual,None,None,None,None,None,EEB AS1-B,None,...,NaT,None,None,MB-AS1-B,179.000,"0,350133544",176.601865,176.800,176.001865,POINT Z (364477.608 6716466.156 0)
3,MB-AS1-A,Sede,Projeto Conceitual,None,None,None,None,None,EEB AS1-A,None,...,NaT,None,None,MB-AS1-A,185.000,"1,20055348",182.155425,182.400,181.405425,POINT Z (363982.717 6717323.103 0)
4,MB-LJ1-A,Sede,Projeto Conceitual,None,None,None,None,None,EEB LJ1-A,None,...,NaT,None,None,MB-LJ1-A,152.000,"0,055451985",149.768621,149.800,149.168621,POINT Z (362054.515 6717579.946 0)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,MB-PE02,Sede,Projeto Conceitual,None,None,None,None,None,EEB PE02,None,...,NaT,None,None,MB-PE02,61.786,"2,655446745",58.710331,59.086,57.860331,POINT Z (362394.839 6710706.856 0)
59,MB-PE03,Sede,Projeto Conceitual,None,None,None,None,None,EEB PE03,None,...,NaT,None,None,MB-PE03,59.754,"0,750241651",57.001162,57.154,56.251162,POINT Z (362240.001 6710596.798 0)
60,MB-AS4,Sede,Projeto Conceitual,None,None,None,None,None,EEB AS4-2,None,...,NaT,None,None,MB-AS4,202.000,"0,452693617",199.543828,199.800,198.943828,POINT Z (365883.466 6717624.545 0)
61,MB-AS16,Sede,Projeto Conceitual,None,None,None,None,None,EEB AS16,None,...,NaT,None,None,MB-AS16,190.000,"0,107661116",187.739076,187.800,187.139076,POINT Z (363422.538 6715066.623 0)


In [15]:
# ----------------------------
# 1) GARANTIR CRS UTM 22S
#    (se seu gdf já tem crs correto, pode pular o set_crs)
# ----------------------------
# Ex.: SIRGAS2000 / UTM 22S -> EPSG:31982
# Ex.: WGS84 / UTM 22S      -> EPSG:32722
# gdf = ...  # seu GeoDataFrame de pontos
# if gdf.crs is None:
#     gdf = gdf.set_crs(epsg=31982)  # ajuste para 32722 se for o seu caso

# ----------------------------
# 2) REPROJETAR PARA WGS84
# ----------------------------
gdf_wgs = gdf.to_crs(epsg=4326).copy()
gdf_wgs["lat"] = gdf_wgs.geometry.y
gdf_wgs["lon"] = gdf_wgs.geometry.x

# ----------------------------
# 3) CONFIGURAR NOMINATIM + RATE LIMIT
# ----------------------------
geolocator = Nominatim(user_agent="bairro_osm_extractor")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)  # gentil com a API

# ----------------------------
# 4) FUNÇÃO PARA EXTRAIR BAIRRO
# ----------------------------
def extrair_bairro(lat, lon):
    try:
        loc = reverse((lat, lon), language="pt")
        if not loc or "address" not in loc.raw:
            return None
        addr = loc.raw["address"]
        # ordem de preferência dos campos que costumam representar "bairro"
        return (
            addr.get("suburb")
            or addr.get("neighbourhood")
            or addr.get("city_district")
            or addr.get("quarter")
            or addr.get("village")
            or addr.get("town")
            or addr.get("city")
        )
    except Exception:
        return None

# ----------------------------
# 5) CACHE: evita repetir chamadas para as mesmas coords
#    (arredonda p/ reduzir duplicatas quase idênticas)
# ----------------------------
def chave_cache(lat, lon, casas=5):
    return (round(lat, casas), round(lon, casas))

coords = gdf_wgs[["lat", "lon"]].copy()
coords["key"] = coords.apply(lambda r: chave_cache(r["lat"], r["lon"]), axis=1)

# Deduplica
coords_uniq = coords.drop_duplicates("key").reset_index(drop=True)

# Consulta com barra de progresso
tqdm.pandas(desc="Buscando bairros (OSM)")
coords_uniq["bairro"] = coords_uniq.progress_apply(
    lambda r: extrair_bairro(r["lat"], r["lon"]), axis=1
)

# Mapeia de volta para todas as linhas
mapa_bairros = dict(zip(coords_uniq["key"], coords_uniq["bairro"]))
gdf_wgs["bairro"] = coords["key"].map(mapa_bairros)

# ----------------------------
# 6) (Opcional) ANEXAR AO GDF ORIGINAL
# ----------------------------
# Se você quer manter o CRS UTM e apenas adicionar a coluna:
gdf_final = gdf.copy()
gdf_final["BAIRRO"] = gdf_wgs["bairro"]

# Resultado:
# gdf_final tem a sua geometria original (UTM 22S) + coluna 'bairro'
print(gdf_final[["BAIRRO", "geometry"]].head())


Buscando bairros (OSM): 100%|██████████| 63/63 [01:04<00:00,  1.03s/it]

                            BAIRRO                            geometry
0                   Sede Municipal  POINT Z (363779.857 6717729.388 0)
1  Loteamento Parque das Palmeiras  POINT Z (364993.158 6717356.479 0)
2                 Linha Santa Cruz  POINT Z (364477.608 6716466.156 0)
3                 Linha Santa Cruz  POINT Z (363982.717 6717323.103 0)
4                         Germânia  POINT Z (362054.515 6717579.946 0)


In [17]:
# salva seu GeoDataFrame em um arquivo .gpkg
gdf_final.to_file("saida.gpkg", driver="GPKG")